# Week 1 - Day 2

Web scraping utility, carried over from Day 1.

In [6]:
# imports

import requests
from bs4 import BeautifulSoup

In [7]:
# Standard headers to fetch a website

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [8]:
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [9]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
F

In [11]:
from openai import OpenAI
ollama = OpenAI(base_url="http://localhost:11434/v1",api_key="1234")

message=[{"role":"system","content":"You are an expert in website summarising, whatever user asks summarise that website in a funny manner"},{
    "role": "user", "content": ed
}]
response = ollama.chat.completions.create(model="llama3.2:1b", messages=message)

print(response.choices[0].message.content)

Let's dive into the world of Artificial Intelligence - aka Edward Donner's Playground (Spoiler Alert: It's a game of LLMs vs. LLMs, aka "The Great Debate Club")

**The Host:** Ed (the enigmatic founder with a calculator for a brain)

**Episode 1: "Introduction to the Arena"**

Welcome to Edward Donner's Playground, where LLMs (Large Language Models) go head-to-head in a battle of wits, cunning, and (dare we say it?) diplomacy. It's a wild ride, folks!

**Episode Highlights:**

* LLMs: "Who's the boss? Me, with my vast knowledge and witty banter!"
* LLMs: "But Ed's still got it – his code is fire, and his analogies are sharp!"
* LLMs: "Time to get strategic! I'll outmaneuver him at every turn!"

**Episode Segments:**

* "C4: The Ultimate Showdown" - LLMs go toe-to-toe in a series of witty exchanges
* "Outsmart: The Art of Deception" - LLMs learn to walk the fine line between being helpful and, well, not-so-helpful
* "Proficiency: Leveling Up" - LLMs perfect their skills, just in time fo

## Trading page Q&A

Scrape a trading/IPO page and ask questions about it, grounded in the scraped content only
(so the model can't just make numbers up - it has to have actually seen them on the page).

In [13]:
def fetch_ipo_page_data(url):
    """
    Fetch an IPO/trading detail page and pull out the structured data tables
    (price band, subscription figures, financials, etc).
    Pages like this bury the real data inside <table> elements after a huge
    nav menu, so grabbing tables directly works much better than raw body text.
    """
    response = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    for irrelevant in soup(["script", "style", "img", "input"]):
        irrelevant.decompose()
    tables = soup.find_all("table")
    table_text = "\n\n".join(t.get_text(separator=" | ", strip=True) for t in tables)
    return f"{title}\n\n{table_text}"[:6_000]

In [14]:
def ask_about_page(url, question, model="llama3.2:1b"):
    """
    Answer a question about a trading page, grounded only in the scraped content.
    """
    content = fetch_ipo_page_data(url)
    system_prompt = (
        "You are an assistant that answers questions about an IPO/trading page. "
        "Only use the page content provided below. If the answer isn't in the "
        "content, say you don't know - do not make anything up.\n\n"
        f"PAGE CONTENT:\n{content}"
    )
    ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    response = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content

In [15]:
# Try it out on a real IPO page

ipo_url = "https://www.chittorgarh.com/ipo/shiprocket-ipo/2450/"

print(ask_about_page(ipo_url, "What is the price band for this IPO?"))

The price band for the Shiprocket IPO is ₹92 to ₹97.


In [16]:
print(ask_about_page(ipo_url, "What is the retail subscription figure, in times?"))

According to the page content, the retail subscription figure is:

48.38 shares of ₹10 per share.


In [ ]:
print(ask_about_page(ipo_url, "What is the lot size and minimum retail investment amount?"))

Chittorgarh is a city and a municipal corporation in the Swain district of the state of Rajasthan in India. It is the district capital and has a rich historical significance, being the erstwhile capital of the independent princely state of Mewar and later serving as a major center of medieval Indian kingdoms, notably the Kachwaha dynasty.
